In [1]:
%pip install soundfile
!pip install -q pesq pystoi
!pip install -q thop
!pip install -q https://github.com/schmiph2/pysepm/archive/master.zip

Note: you may need to restart the kernel to use updated packages.
  Preparing metadata (setup.py) ... done
     \ 1.8 MB 15.7 MB/s 0:00:00m
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     \ 223.8 kB 3.1 MB/s 0:00:00
  Preparing metadata (setup.py) ... done
     - 39.3 kB 106.9 MB/s 0:00:00
  Preparing metadata (setup.py) ... done
     \ 59.4 MB 111.3 MB/s 0:00:03
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 3.4 MB/s eta 0:00:0000:01


### Declare Libraries

In [2]:
from __future__ import annotations
from typing import Any, Dict, List, Optional, Tuple, Iterator, Sequence
from pathlib import Path
import json
import re
import wave
import numpy as np
import soundfile as sf
import os

import torch
from torch import Tensor, nn
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch.utils.data import DataLoader, Dataset
from torch.utils.data import BatchSampler
import time

from collections import defaultdict
import matplotlib.pyplot as plt
import random
import shutil
import math
import glob

## ERB filter bank, STFT, Inverse STFT

In [3]:
def spec_to_wav(spec: Tensor, n_fft: int = 512, hop_length: int = 128, win_length: int = 512, window: Optional[Tensor] = None) -> Tensor:
    """Convert complex spectrogram [B, T, F] back to time-domain waveform [B, N-sampels]"""
    if window is None:
        window = torch.hann_window(win_length, device=spec.device)

    # torch.istft expects shape [B, F, T]
    spec_transposed = spec.transpose(-1, -2).contiguous()
    wav = torch.istft(spec_transposed, n_fft=n_fft, hop_length=hop_length, win_length=win_length, window=window, center=True)
    return wav

def load_mono(path, target_sr=16000):
    # Read the audio
    wav_np, sr = sf.read(path, dtype='float32')
    wav = torch.from_numpy(wav_np).float() 
    
    if wav.ndim == 2:
        wav = wav.mean(dim=1)
        
    # Resample the audio if it is not 16,000 Hz!
    if sr != target_sr:
        wav = torchaudio.functional.resample(wav, orig_freq=sr, new_freq=target_sr)
        
    return wav, target_sr

def stft_magnitude(wav, device, n_fft=512, hop_length=128, win_length=512):
    window = torch.hann_window(win_length, device=device)
    wav = wav.to(device)

    spec = torch.stft(
        wav,
        n_fft=n_fft,
        hop_length=hop_length,
        win_length=win_length,
        window=window,
        center=True,
        return_complex=True,
    )

    # [freq, time] -> [time, freq]
    return spec.transpose(-1, -2)

def hz_to_erb(freq):
    return 21.4 * torch.log10(1.0 + 0.00437 * freq)

def erb_to_hz(erb):
    return (10 ** (erb / 21.4) - 1.0) / 0.00437

def make_erb_filterbank(sample_rate, n_fft, erb_bins, low_freq=0.0, high_freq=None):
    if high_freq is None:
        high_freq = sample_rate / 2

    erb_edges = torch.linspace(
        hz_to_erb(torch.tensor(low_freq)),
        hz_to_erb(torch.tensor(high_freq)),
        erb_bins + 2,
    )

    hz_edges = erb_to_hz(erb_edges)
    fft_freqs = torch.linspace(0.0, sample_rate / 2, n_fft // 2 + 1)

    filterbank = torch.zeros(erb_bins, n_fft // 2 + 1)

    for i in range(erb_bins):
        left = hz_edges[i]
        center = hz_edges[i + 1]
        right = hz_edges[i + 2]

        rising = (fft_freqs - left) / (center - left)
        falling = (right - fft_freqs) / (right - center)

        filterbank[i] = torch.minimum(rising, falling).clamp_min(0.0)
    return filterbank

## Loss function

In [4]:
"""Return normalized inverse ERB weights with shape ``[F, E]``."""
def erb_synthesis_matrix(filterbank: Tensor, eps: float = 1e-8) -> Tensor:
    return filterbank.T / filterbank.sum(dim=0, keepdim=True).T.clamp_min(eps)

def apply_erb_gains(input_spec: Tensor, gains: Tensor, synthesis_matrix: Tensor) -> Tensor:
    if gains.ndim != 4 or gains.shape[1] != 1:
        raise ValueError("gains must have shape [B, 1, T, E]")
    """synthesis_matrix: [F, E], which converts values defined on 
    ERB bands back into values for individual FFT frequency bins.
       gains[:, 0]: makes [B, 1, T, E] -> [B, T, E]"""
    frequency_gain = gains[:, 0] @ synthesis_matrix.T    # -> [B, T, F]
    return input_spec * frequency_gain.to(input_spec.dtype)

### This is a derivative of arctan(b, a), where z = a + bi
$$
\delta X = \delta \phi \cdot \left( \frac{-\Im\{X\}}{|X_h|^2}, \frac{\Re\{X\}}{|X_h|^2} \right)
$$

This class is a custom, numerically stable PyTorch function used to calculate the phase (angle) of complex numbers—which is a very common operation when dealing with audio spectrograms (STFT).

Its main purpose is to prevent your model from crashing or outputting NaN (Not a Number) during training when it encounters pure silence.

In [5]:
class SafeAngle(torch.autograd.Function):
    def forward(ctx, x: Tensor) -> Tensor:
        ctx.save_for_backward(x)
        return torch.atan2(x.imag, x.real)

    def backward(ctx, grad: Tensor) -> Tensor:
        (x,) = ctx.saved_tensors
        # |X_h|^2 = max(Re^2 + Im^2, 1e-12)
        denom = (x.real.square() + x.imag.square()).clamp_min(1e-12)
        grad_scaled = grad / denom
        return torch.view_as_complex(torch.stack([-x.imag * grad_scaled, x.real * grad_scaled], dim=-1))

safe_angle = SafeAngle.apply

### 1. Compressed Spectral Loss and ERB Mask Loss

In [6]:
class EnhancedSpectralLoss(nn.Module):
    def __init__(self, power: float = 0.6, factor_magnitude: float = 1.0, factor_complex: float = 1.0, factor_under: float = 2.0):
        super().__init__()
        self.power = power
        self.factor_magnitude = factor_magnitude
        self.factor_complex = factor_complex
        self.factor_under = factor_under

    def forward(self, predicted_spec: Tensor, target_spec: Tensor):
        pred_mag = predicted_spec.abs().clamp_min(1e-8).pow(self.power)
        target_mag = target_spec.abs().clamp_min(1e-8).pow(self.power)

        mag_err = (pred_mag - target_mag).square()
        if self.factor_under != 1.0:
            # Overpenalize when prediction < target (speech was cut)
            under_weight = torch.where(pred_mag < target_mag, self.factor_under, 1)
            mag_err = mag_err * under_weight

        loss_mag = mag_err.mean() * self.factor_magnitude

        loss_complex = torch.tensor(0.0, device=predicted_spec.device)
        if self.factor_complex > 0:
            pred_c = pred_mag * torch.exp(1j * safe_angle(predicted_spec))
            target_c = target_mag * torch.exp(1j * safe_angle(target_spec))
            loss_complex = F.mse_loss(torch.view_as_real(pred_c), torch.view_as_real(target_c)) * self.factor_complex

        return loss_mag + loss_complex

class ERBMaskLoss(nn.Module):
    """
    Direct supervision on the ERB mask using Ideal Amplitude Mask (IAM)
    Compute the loss of output gains with ideal gains/IAM
    Uses L2 + L4 penalties to heavily penalize large gain errors
    """
    def __init__(self, power_compress: float = 0.6, factor_under: float = 2.0, factor_l2: float = 1.0, factor_l4: float = 10.0, eps: float = 1e-12):
        super().__init__()
        self.power_compress = power_compress
        self.factor_under = factor_under
        self.factor_l2 = factor_l2
        self.factor_l4 = factor_l4
        self.eps = eps

    def forward(self, predicted_gains: Tensor, input_spec: Tensor, target_spec: Tensor, filterbank: Tensor) -> Tensor:
        input_pow = input_spec.abs().square()
        target_pow = target_spec.abs().square()

        fb = filterbank.to(input_spec.device).T
        input_erb_pow = input_pow @ fb
        target_erb_pow = target_pow @ fb

        # Ideal Amplitude Mask in ERB domain: sqrt(S_power / (X_power + eps))
        target_mask = (target_erb_pow / (input_erb_pow + self.eps)).sqrt().clamp(0.0, 1.0)
        target_mask = target_mask.unsqueeze(1) # [B, 1, T, E]

        # Power compress target and prediction for dynamic range equalization
        g_pred = predicted_gains.clamp_min(self.eps).pow(self.power_compress)
        g_true = target_mask.clamp_min(self.eps).pow(self.power_compress)

        diff = g_pred - g_true
        if self.factor_under != 1.0:
            diff = diff * torch.where(g_pred < g_true, self.factor_under, 1.0)

        # L2 and L4 norm penalty
        loss_l2 = diff.square().clamp_min(1e-13).mean() * self.factor_l2
        loss_l4 = diff.square().clamp_min(1e-13).pow(2).mean() * self.factor_l4

        return loss_l2 + loss_l4

### 2. Combined Loss

In [9]:
class DeepFilterNetCombinedLoss(nn.Module):
    def __init__(
        self,
        filterbank: Tensor,
        lambda_mask: float = 0.5,
        lambda_stage1: float = 0.5,
    ):
        super().__init__()
        self.spectral_loss = EnhancedSpectralLoss(power=0.3, factor_under=2.0)
        self.mask_loss = ERBMaskLoss(power_compress=0.3, factor_under=2.0)

        self.lambda_stage1 = lambda_stage1
        self.lambda_mask = lambda_mask
        self.register_buffer("filterbank", filterbank)

    def forward(self,
                stage1_spec: Tensor, # [B, T, F] (Y^G)
                target_spec: Tensor, # [B, T, F] Clean target (S)
                input_spec: Tensor, # [B, T, F] Noisy input (X)
                pred_gains: Tensor, # [B, 1, T, erb_bins]
               ) -> Tensor:

        l_stage1 = self.spectral_loss(stage1_spec, target_spec)
        
        l_mask = self.mask_loss(pred_gains, input_spec, target_spec, self.filterbank)
        
        
        total_loss = self.lambda_mask * l_mask + self.lambda_stage1 * l_stage1

        return total_loss

## Set up layers

In [11]:
def channel_shuffle(x: Tensor, groups: int) -> Tensor:
    b, t, d = x.shape
    if d % groups != 0:
        raise ValueError("Feature dimension must be divisible by groups")

    features_per_group = d // groups
    x = x.view(b, t, groups, features_per_group)
    x = x.transpose(2, 3).contiguous()
    x = x.view(b, t, d)

    return x

class SeparableConv2d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, *, kernel_size: Tuple[int, int] = (3, 2), stride: Tuple[int, int] = (1, 1), lookahead: int = 0) -> None:
        super().__init__()

        kt, kf = kernel_size

        if lookahead < 0 or lookahead > kt - 1:
            raise ValueError("lookahead must satisfy 0 <= lookahead <= kt - 1")

        time_left = kt - 1 - lookahead
        time_right = lookahead
        self.pad = (
            kf // 2, # left
            kf - 1 - kf // 2, # right
            time_left, # top
            time_right, # bottom
        )

        self.depthwise = nn.Conv2d(
            in_channels=in_channels,
            out_channels=in_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=0,
            groups=in_channels,
            bias=False,
        )

        self.pointwise = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1,
            bias=False,
        )

        self.norm = nn.BatchNorm2d(out_channels)
        self.act = nn.ReLU()
    def forward(self, x: Tensor) -> Tensor:
        x = F.pad(x, self.pad)
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.norm(x)
        x = self.act(x)
        return x
    
class SeparableTConv2d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, *, scale_factor: int = 2, lookahead: int = 0) -> None:
        super().__init__()
        self.scale_factor = scale_factor

        self.conv = SeparableConv2d(in_channels, out_channels, kernel_size=(3, 2), lookahead=lookahead)

    def forward(self, x: Tensor) -> Tensor:
        x = F.interpolate(
            x, 
            scale_factor=(1, self.scale_factor), # [width, height] ~ [time, frequency]
            mode="nearest"
        )
        return self.conv(x)

class GroupedLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, groups: int = 8, shuffle: bool = True) -> None:
        super().__init__()

        if in_features % groups:
            raise ValueError("in_features must be divisible by groups")

        if out_features % groups:
            raise ValueError("out_features must be divisible by groups")

        self.groups = groups
        self.shuffle = shuffle

        self.in_per_group = in_features // groups
        self.out_per_group = out_features // groups

        self.layers = nn.ModuleList(
            [
                nn.Linear(
                    self.in_per_group,
                    self.out_per_group,
                )
                for _ in range(groups)
            ]
        )

    def forward(self, x: Tensor) -> Tensor:
        # x: [B, T, D]

        chunks = x.split(self.in_per_group, dim=-1)

        output_chunks = []

        for layer, chunk in zip(self.layers, chunks):
            y = layer(chunk)
            output_chunks.append(y)

        x = torch.cat(output_chunks, dim=-1)

        if self.shuffle:
            x = channel_shuffle(x, self.groups)

        return x

class GroupedGRU(nn.Module):
    def __init__(self, input_size: int = 512, hidden_size: int = 512, groups: int = 8, shuffle: bool = True) -> None:
        super().__init__()
        if input_size % groups:
            raise ValueError("input_size must be divisible by groups")
        if hidden_size % groups:
            raise ValueError("hidden_size must be divisible by groups")

        self.groups = groups
        self.shuffle = shuffle

        self.input_per_group = input_size // groups
        self.hidden_per_groups = hidden_size // groups

        self.grus = nn.ModuleList(
            [
                nn.GRU(
                    input_size=self.input_per_group,
                    hidden_size=self.hidden_per_groups,
                    batch_first=True,
                )
                for _ in range(groups)
            ]
        )


    def forward(self, x: Tensor) -> Tensor:
        # [B, T, D]
        chunks = x.split(self.input_per_group, dim=-1)
        outputs = []

        for gru, chunk in zip(self.grus, chunks):
            # GRU returns [output, hidden]
            y, _ = gru(chunk)
            outputs.append(y)

        x = torch.cat(outputs, dim=-1)

        if self.shuffle:
            x = channel_shuffle(x, self.groups)

        return x

class GroupedGRUStack(nn.Module):
    def __init__(self, size: int = 512, groups: int = 8, num_layers: int = 3) -> None:
        super().__init__()

        self.layers = nn.ModuleList(
            [
                GroupedGRU(
                    input_size=size,
                    hidden_size=size,
                    groups=groups,
                    shuffle=True,
                )
                for _ in range(num_layers)
            ]
        )

    def forward(self, x: Tensor) -> Tensor:
        for layer in self.layers:
            x = layer(x)

        return x

class PConv(nn.Module):
    def __init__(self, channels: int = 64, out_channels: int = 64) -> None:
        super().__init__()

        self.conv = nn.Conv2d(channels, out_channels, kernel_size=1, bias=False)

    def forward(self, x: Tensor) -> Tensor:
        return self.conv(x)

## Implement ERB encoder, decoder

In [12]:
class ERBEncoder(nn.Module):
    def __init__(self, erb_bins: int = 32, channels: int = 64, hidden_size: int = 512, groups: int = 8, conv_lookahead: int = 2) -> None:
        super().__init__()

        if erb_bins % 8:
            raise ValueError("erb_bins must be divisible by 8")

        self.erb_bins = erb_bins
        self.channels = channels
        self.conv_lookahead = conv_lookahead
        lookahead0 = 1 if conv_lookahead > 0 else 0
        lookahead1 = 1 if conv_lookahead > 1 else 0
        lookahead2 = 1 if conv_lookahead > 2 else 0
        self.conv0 = SeparableConv2d(1, channels, lookahead=lookahead0)
        self.conv1 = SeparableConv2d(channels, channels, stride=(1, 2), lookahead=lookahead1) # B -> B / 2
        self.conv2 = SeparableConv2d(channels, channels, stride=(1, 2), lookahead=lookahead2)
        self.conv3 = SeparableConv2d(channels, channels, stride=(1, 2), lookahead=0)
        self.glinear = GroupedLinear(in_features=channels*erb_bins//8, out_features=hidden_size, groups=groups)
        self.gru = GroupedGRUStack(size=hidden_size, groups=groups, num_layers=3)
        self.dropout = nn.Dropout(p=0.3)

    def forward(self, x: Tensor) -> Tuple[Tensor, Tensor, Tensor, Tensor, Tensor]:
        x0 = self.conv0(x)
        x1 = self.conv1(x0)
        x2 = self.conv2(x1)
        x3 = self.conv3(x2)

        batch, channels, time, freq = x3.shape
        x = x3.permute(0, 2, 1, 3)
        x = x.reshape(batch, time, channels*freq)

        x = self.glinear(x)
        embedding = self.gru(x)
        embedding = self.dropout(embedding)

        return x0, x1, x2, x3, embedding

class ERBDecoder(nn.Module):
    def __init__(self, channels: int = 64, erb_bins: int = 32, hidden_size: int = 512) -> None:
        super().__init__()

        bottleneck_freq = erb_bins // 8
        bottleneck_size = channels * bottleneck_freq

        self.channels = channels
        self.bottelneck_freq = bottleneck_freq

        self.linear = GroupedLinear(hidden_size, bottleneck_size)

        self.p3 = PConv(channels)
        self.p2 = PConv(channels)
        self.p1 = PConv(channels)
        self.p0 = PConv(channels)

        self.up3 = SeparableTConv2d(channels, channels, lookahead=0)
        self.up2 = SeparableTConv2d(channels, channels, lookahead=0)
        self.up1 = SeparableTConv2d(channels, channels, lookahead=0)

        self.final_conv = SeparableConv2d(channels, 1, kernel_size=(3, 2), lookahead=0)
        self.output_activation = nn.Sigmoid()
        self.dropout = nn.Dropout(p=0.3)

    def forward(self, embedding: Tensor, x0: Tensor, x1: Tensor, x2: Tensor, x3: Tensor) -> Tensor:
        batch, time, _ = embedding.shape

        x = self.linear(embedding)
        x = self.dropout(x)

        x = x.reshape(batch, time, self.channels, self.bottelneck_freq)
        x = x.permute(0, 2, 1, 3).contiguous()

        p3_x3 = self.p3(x3)
        x = x + p3_x3

        x = self.up3(x)
        x = x + self.p2(x2)

        x = self.up2(x)
        x = x + self.p1(x1)

        x = self.up1(x)
        x = x + self.p0(x0)

        gains = self.final_conv(x)
        gains = self.output_activation(gains)

        return gains

## Deep Neural Network (DNN)

In [13]:
class DeepFilterNetDNN(nn.Module):
    def __init__(self, erb_bins: int = 32, channels: int = 64, hidden_size: int = 512, groups: int = 8, conv_lookahead: int = 2) -> None:
        super().__init__()
        
        self.encoder = ERBEncoder(
            erb_bins=erb_bins,
            channels=channels,
            hidden_size=hidden_size,
            groups=groups,
            conv_lookahead=conv_lookahead,
        )

        self.decoder = ERBDecoder(
            channels=channels,
            erb_bins=erb_bins,
            hidden_size=hidden_size
        )

    def forward(self, erb_features: Tensor) -> Tensor:
        # Stage 1:
        e0, e1, e2, e3, embedding = self.encoder(erb_features)
        gains = self.decoder(embedding, e0, e1, e2, e3)

        return gains


## On-the-fly augmentation - Dataset Generator
Instead of pre-mixing and saving thousands of noisy audio files to the hard drive (which would take up terabytes of storage), this section defines a dynamic dataset generator. During training, as the neural network requests a new batch of data, the generator dynamically creates unique audio samples "on-the-fly" by:

1. Randomly selecting a clean speech file.
2. Applying a random Room Impulse Response (RIR) to simulate physical environments.
3. Mixing in a random noise file at varying Signal-to-Noise Ratios (SNR).

This approach saves massive amounts of disk space and ensures the AI model rarely sees the exact same acoustic combination twice, acting as powerful data augmentation to prevent overfitting.

In [37]:
class DynamicMixingDataset(Dataset):
    def __init__(self, clean_dir, noise_dir, rir_dir, segment_samples=48000, flag: bool = False):
        self.clean_files = sorted(glob.glob(f"{clean_dir}/**/*.wav", recursive=True))
        self.noise_files = sorted(glob.glob(f"{noise_dir}/**/*.wav", recursive=True))

        if rir_dir is not None:
            self.rir_files = sorted(glob.glob(f"{rir_dir}/**/*.wav", recursive=True))
        else:
            self.rir_files = []

        self.segment_samples = segment_samples
        self.filterbank = filterbank
        self.target_dB_FS = -25

        # Random volume variation (+/- 10dB) 
        # => +10 simulates a user who is shouting or holding a microphone very close to the mounth
        # => -10 simulates a user who is whispering, or using a smart speaker placed on a table across a room
        self.target_db_FS_floating_value = 10

        # 50 ms threshold for early reflections
        self.time_early = 0.05

        self.snr_levels = [-5.0, 0.0, 5.0, 10.0, 15.0, 20.0]
        self.flag = flag

    def __len__(self):
        return len(self.clean_files)

    def _load_and_crop(self, path, rng, target_sr=16000, target_samples=48000):
        # Read directly from the path!
        wav_np, sr = sf.read(path, dtype='float32')
        wav = torch.from_numpy(wav_np).float()
        
        if wav.ndim == 2:
            wav = wav.mean(dim=1)
            
        orig_length = int(target_samples * (sr / target_sr))
        
        # Crop first
        if wav.shape[-1] > orig_length:
            start = rng.randint(0, wav.shape[-1] - orig_length)
            wav = wav[start : start + orig_length]
        else:
            wav = torch.nn.functional.pad(wav, (0, orig_length - wav.shape[-1]))
            
        # Resample second
        if sr != target_sr:
            wav = torchaudio.functional.resample(wav, orig_freq=sr, new_freq=target_sr)
            
        return wav.unsqueeze(0)

    def _tailor_dB_FS(self, wav, target_dB_FS, eps=1e-6):
        """Normalizes audio to a specific dBFS level"""
        rms = torch.sqrt(torch.mean(wav**2) + eps)
        scalar = 10 ** (target_dB_FS / 20) / rms
        wav = wav * scalar
        return wav, rms, scalar

    def _is_clipped(self, wav):
        """Check if audio is clipping"""
        return torch.max(torch.abs(wav)) > 0.99

    def _apply_rir_and_target(self, clean_y, rir_wav, sr=16000):
        """Split RIR into full reverb (input) and early reflections (target)"""
        T0 = int(self.time_early * sr)
        T1 = 480   # Smooth transition window
        len_rir = rir_wav.shape[-1]
        device = rir_wav.device

        # 1. Full Reverberation (What the microphone hear)
        clean_y_reverb = torchaudio.functional.fftconvolve(clean_y, rir_wav)
        clean_y_reverb = clean_y_reverb[:, :clean_y.shape[-1]]

        # 2. Early Reflections only (What the model should output)
        if len_rir > (T0 + T1):
            Rd = 200.0
            alpha = 0.4

            # Decay Component
            in_slope_decay = torch.arange(start=1, end=len_rir - T0 + 1, step=1, device=device, dtype=torch.float32)
            slope_decay = 10 ** ((-3.0 * in_slope_decay) / Rd)
            decay = torch.cat((torch.ones(T0, device=device), slope_decay), dim=0)

            # Attenuated Component (Cosine smoothing)
            in_slope_att = torch.arange(start=1, end=T1 + 1, step=1, device=device, dtype=torch.float32)
            # The cosine transition curve
            slope_att = (1 + alpha) / 2 + (1 - alpha) * torch.cos(torch.pi * in_slope_att / T1) / 2
            const = alpha * torch.ones(len_rir - T1 - T0, device=device)
            attenuated = torch.cat((torch.ones(T0, device=device), slope_att, const), dim=0)

            # Combine them
            out_filter = attenuated * decay
            rir_shorten = rir_wav * out_filter.unsqueeze(0)

        elif (T0 + T1) >= len_rir > T0:
            Rd = 200.0
            in_slope_decay = torch.arange(1, len_rir - T0 + 1, 1, device=device, dtype=torch.float32)
            slope_decay = 10 ** ((-3.0 * in_slope_decay) / Rd)
            decay = torch.cat((torch.ones(T0, device=device), slope_decay), dim=0)
            rir_shorten = rir_wav * decay.unsqueeze(0)

        else:
            rir_shorten = rir_wav

        clean_target = torchaudio.functional.fftconvolve(clean_y, rir_shorten)
        clean_target = clean_target[:, :clean_y.shape[-1]]

        return clean_y_reverb, clean_target

    def snr_mix(self, clean_y, noise_y, snr, rir_wav, rng, eps=1e-6, sr=16000):
        """Advanced mixing with rigorous dBFS normalization"""
        if rir_wav is not None:
            clean_y_reverb, clean_target = self._apply_rir_and_target(clean_y, rir_wav, sr)
        else:
            clean_y_reverb = clean_y
            clean_target = clean_y

        # Normalize noth to the exact same target dBFS to establish a baseline
        clean_y_reverb, clean_reverb_rms, _ = self._tailor_dB_FS(clean_y_reverb, self.target_dB_FS, eps)
        noise_y, noise_rms, _ = self._tailor_dB_FS(noise_y, self.target_dB_FS, eps)

        # Scale noise to match the requested SNR
        snr_scalar = clean_reverb_rms / (10 ** (snr / 20)) / (noise_rms + eps)
        noise_y = noise_y * snr_scalar
        noisy_y = clean_y_reverb + noise_y

        # Randomly select a final target RMS in dBFS (simulating different microphone gains)
        noisy_target_dB_FS = rng.randint(
            self.target_dB_FS - self.target_db_FS_floating_value,
            self.target_dB_FS + self.target_db_FS_floating_value
        )
        noisy_y, _, noisy_scalar = self._tailor_dB_FS(noisy_y, noisy_target_dB_FS, eps)

        # Apply the exact same final scalar to the target so the model learns the correct output volume
        clean_target = clean_target * noisy_scalar

        if self._is_clipped(noisy_y):
            noisy_y_scalar = torch.max(torch.abs(noisy_y)) / (0.99 - eps)
            noisy_y = noisy_y / noisy_y_scalar
            clean_target = clean_target / noisy_y_scalar

        return noisy_y, clean_target
    
    def __getitem__(self, idx):
        if self.flag:
            rng = random.Random(42 + idx)
        else:
            rng = random.Random()
        
        clean_wav = self._load_and_crop(self.clean_files[idx], rng, target_samples=self.segment_samples)

        # Apply RIR (Reverberation) 50% of the time
        rir_wav = None
        if len(self.rir_files) > 0:
            rir_idx = rng.randint(0, len(self.rir_files) - 1)
            rir_wav = self._load_and_crop(self.rir_files[rir_idx], rng, target_samples=16000)

        """Create complex noise mixture (up to 5 noise signals)"""
        noise_mixture = torch.zeros_like(clean_wav)
        num_noises = rng.randint(1, 5)

        for _ in range(num_noises):
            n_idx = rng.randint(0, len(self.noise_files) - 1)
            n_wav = self._load_and_crop(self.noise_files[n_idx], rng, target_samples=self.segment_samples)
            noise_mixture += n_wav

        """Advanced SNR Mixing and Dereverberation Targeting"""
        target_snr_db = rng.choice(self.snr_levels)
        noisy_wav, clean_target = self.snr_mix(clean_wav, noise_mixture, target_snr_db, rir_wav, rng)

        return {
            "noisy_wav": noisy_wav.squeeze(),    # Shape: [T]
            "clean_target": clean_target.squeeze() # Shape: [T]
        }

## Define Parameters

In [59]:
sample_rate = 16000
n_fft = 512
hop_length = 128
win_length = 512
erb_bins = 32
batch_size = 40
epochs = 20
learning_rate = 1e-4
df_bins = 96
df_order = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
filterbank = make_erb_filterbank(sample_rate=sample_rate, n_fft=n_fft, erb_bins=erb_bins)
criterion = DeepFilterNetCombinedLoss(filterbank=filterbank).to(device)


## Dataloader

In [44]:
train_dataset = DynamicMixingDataset(
    clean_dir="/kaggle/input/datasets/quanninhhoang/dns-cleanspeech-dataset/train",
    noise_dir="/kaggle/input/datasets/quanninhhoang/dns-noise-dataset/train",
    rir_dir="/kaggle/input/datasets/quanninhhoang/dns-impulse-dataset/train",
)

val_dataset = DynamicMixingDataset(
    clean_dir="/kaggle/input/datasets/quanninhhoang/dns-cleanspeech-dataset/val",
    noise_dir="/kaggle/input/datasets/quanninhhoang/dns-noise-dataset/val",
    rir_dir="/kaggle/input/datasets/quanninhhoang/dns-impulse-dataset/val",
    flag=True
)

test_dataset = DynamicMixingDataset(
    clean_dir="/kaggle/input/datasets/quanninhhoang/dns-cleanspeech-dataset/test",
    noise_dir="/kaggle/input/datasets/quanninhhoang/dns-noise-dataset/test",
    rir_dir=None,
    segment_samples=160000,
    flag=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)


## Exponential Moving Average (EMA) Normalization 

In [ ]:
class CausalEmaNorm(nn.Module):
    def __init__(self, alpha: float = 0.99, eps: float = 1e-5):
        """
        Args: The decay factor. 0.99 roughly corresponds to a 1-second memory for a 10ms hop size
        """
        super().__init__()
        self.alpha = alpha
        self.eps = eps

    def forward(self, x: torch.Tensor):
        # Shape: [Batch, Channels, Time, Freq]
        B, C, T, F = x.shape

        out = torch.empty_like(x)
        
        # Initialize the moving mean and variance with the first frame.
        mean = x[:, :, 0, :]
        var = torch.zeros_like(mean)
        
        # The first frame is simply centered to 0 (variance is 0 so far)
        out[:, :, 0, :] = (x[:, :, 0, :] - mean) / torch.sqrt(var + self.eps)
        
        for t in range(1, T):
            curr_x = x[:, :, t, :]
            diff = curr_x - mean
            mean = self.alpha * mean + (1 - self.alpha) * curr_x
            var = self.alpha * var + (1 - self.alpha) * diff * (curr_x - mean)
            
            out[:, :, t, :] = (curr_x - mean) / torch.sqrt(var + self.eps)
            
        return out
            

## Validation Function

In [62]:
def evaluate_model(model, loader, synthesis_matrix, device, filterbank, criterion):
    model.eval()
    total_val_loss = 0.0
    total_batches = 0

    with torch.no_grad():
        ema_norm = CausalEmaNorm(alpha=0.99, eps=1e-5).to(device)
        window = torch.hann_window(512).to(device)
        for batch in loader:
            noisy_wav = batch["noisy_wav"].to(device)       
            clean_target = batch["clean_target"].to(device)
            
            input_spec = torch.stft(noisy_wav, n_fft=512, hop_length=128, win_length=512, window=window, return_complex=True)
            target_spec = torch.stft(clean_target, n_fft=512, hop_length=128, win_length=512, window=window, return_complex=True)
            
            input_erb = torch.matmul(input_spec.abs().square().transpose(1, 2), filterbank.T)
            input_erb = torch.log10(input_erb + 1e-7).unsqueeze(1)
            
            input_spec = input_spec.transpose(1, 2)
            target_spec = target_spec.transpose(1, 2)

            input_erb = ema_norm(input_erb)

            with torch.amp.autocast(device.type):
                gains = model(input_erb)
                stage1_spec = apply_erb_gains(input_spec, gains, synthesis_matrix)
                loss = criterion(
                    stage1_spec=stage1_spec,
                    target_spec=target_spec,
                    input_spec=input_spec,
                    pred_gains=gains,
                )
                
            total_val_loss += loss.item()
            total_batches += 1

    return total_val_loss / max(total_batches, 1)


## Training Step

In [66]:
def train_model(model, train_loader, val_loader, synthesis_matrix, optimizer, scheduler, criterion, epochs, checkpoint_dir, filterbank, device):
    checkpoint_dir = Path(checkpoint_dir)
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    best_val_loss = float("inf")
    history = []

    window = torch.hann_window(512).to(device)
    filterbank = filterbank.to(device)
    scaler = torch.amp.GradScaler(device.type)
    ema_norm = CausalEmaNorm(alpha=0.99, eps=1e-5).to(device)
    for epoch in range(epochs):
        epoch_start_time = time.perf_counter()
        model.train()

        total_train_loss = 0.0
        total_batches = 0
        for batch in train_loader:
            noisy_wav = batch["noisy_wav"].to(device)       
            clean_target = batch["clean_target"].to(device) 
            
            input_spec = torch.stft(noisy_wav, n_fft=512, hop_length=128, win_length=512, window=window, return_complex=True)
            target_spec = torch.stft(clean_target, n_fft=512, hop_length=128, win_length=512, window=window, return_complex=True)
            
            input_erb = torch.matmul(input_spec.abs().square().transpose(1, 2), filterbank.T)
            input_erb = torch.log10(input_erb + 1e-7).unsqueeze(1)

            input_erb = ema_norm(input_erb)
            
            input_spec = input_spec.transpose(1, 2)
            target_spec = target_spec.transpose(1, 2)
            
            optimizer.zero_grad(set_to_none=True)

            # NaN Handling and Gradient Clipping
            try:
                with torch.amp.autocast(device.type):
                    gains = model(input_erb)
                    stage1_spec = apply_erb_gains(input_spec, gains, synthesis_matrix)
                    loss = criterion(
                        stage1_spec=stage1_spec,
                        target_spec=target_spec,
                        input_spec=input_spec,
                        pred_gains=gains,
                    )

                # Scale the loss and call backward to compute scaled gradients
                scaler.scale(loss).backward()
    
                # Unscale the gradients of optimizer's assigned params in-place.
                scaler.unscale_(optimizer)
    
                # Clip gradients to prevent exploding gradients (crucial for GRUs)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0, error_if_nonfinite=True)
    
                # Step optimizer and update scaler
                scaler.step(optimizer)
                scaler.update()
                total_train_loss += loss.detach().item()
                total_batches += 1
                
                # loss.backward()
                # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                # optimizer.step()
                # total_train_loss += loss.detach().item()
                # total_batches += 1
        
            except RuntimeError as e:
                # Catch NaN/Inf gradients gracefully to prevent training crashes
                if "nan" in str(e).lower() or "non-finite" in str(e).lower():
                    print(f"Skipping batch due to non-finite gradients: {e}")
                    optimizer.zero_grad(set_to_none=True)
                    continue
                else:
                    raise e
            
        train_loss = (total_train_loss / max(total_batches, 1))
        
        val_loss = evaluate_model(
            model=model,
            loader=val_loader,
            synthesis_matrix=synthesis_matrix,
            device=device,
            filterbank=filterbank
        )

        if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
            scheduler.step(val_loss)
        else:
            scheduler.step()

        epoch_duration = time.perf_counter() - epoch_start_time
        mins, secs = divmod(int(epoch_duration), 60)
        time_str = f"{mins:02d}m {secs:02d}s"
        
        history.append(
            {
                "epoch": epoch+1,
                "train_loss": train_loss,
                "val_loss": val_loss,
            }
        )
        epoch_checkpoint = checkpoint_dir / (f"epoch_{epoch + 1:03d}.ckpt")
        torch.save(
            {
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scaler_state_dict": scaler.state_dict(),
                "train_loss": train_loss,
                "val_loss": val_loss,
            },
            epoch_checkpoint,
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_checkpoint = checkpoint_dir / (f"best_model.ckpt")
            torch.save(
                {
                    "epoch": epoch + 1,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "scaler_state_dict": scaler.state_dict(),
                    "train_loss": train_loss,
                    "val_loss": val_loss,
                },
                best_checkpoint,
            )
            print(
                f"Epoch {epoch + 1:02d}/{epochs:02d} [{time_str}] "
                f"train={train_loss:.6f} "
                f"val={val_loss:.6f} "
                f"best checkpoint saved",
                flush=True,
            )
        else:
            print(
                f"Epoch {epoch + 1:02d}/{epochs:02d} [{time_str}] "
                f"train={train_loss:.6f} "
                f"val={val_loss:.6f}",
                flush=True,
            )

    return history


In [64]:
from torch.optim.lr_scheduler import ReduceLROnPlateau

model = DeepFilterNetDNN(erb_bins=erb_bins).to(device)
synthesis_matrix = erb_synthesis_matrix(filterbank).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

In [67]:
checkpoint_dir = "/kaggle/working/erb_checkpoints"

history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    synthesis_matrix=synthesis_matrix,
    optimizer=optimizer,
    scheduler=scheduler,
    criterion=criterion,
    epochs=epochs,
    checkpoint_dir=checkpoint_dir,
    filterbank=filterbank,
    device=device
)

KeyboardInterrupt: 

## Visualize Training and Validation Loss

In [ ]:
epochs_plot = [
    item["epoch"]
    for item in history
]

train_loss = [
    item["train_loss"]
    for item in history
]

val_loss = [
    item["val_loss"]
    for item in history
]

plt.figure(figsize=(8, 5))

plt.plot(
    epochs_plot,
    train_loss,
    color="red",
    marker="o",
    label="Training Loss"
)
plt.plot(
    epochs_plot,
    val_loss,
    color="blue",
    marker="s",
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Testing Step - Compute standard speech metrices (PESQ, STOI, SI-SDR)

### 1. Using Standard Speech Metrices (PESQ, STOI, SI-SDR)

In [ ]:
try:
    import pystoi
except ImportError:
    pystoi = None

try:
    from pesq import pesq
except ImportError:
    pesq = None

def calculate_si_sdr(estimate: Tensor, reference: Tensor, eps: float = 1e-8) -> float:
    est = estimate.flatten().double()
    ref = reference.flatten().double()

    # Zero-mean
    est = est - est.mean()
    ref = ref - ref.mean()

    # Projection scale factor alpha 
    ref_energy = torch.sum(ref ** 2) + eps
    alpha = torch.sum(est * ref) / ref_energy

    e_target = alpha * ref
    e_res = est - e_target

    target_pow = torch.sum(e_target ** 2) + eps
    res_pow = torch.sum(e_res ** 2) + eps

    return float(10 * torch.log10(target_pow / res_pow).item())

def calculate_stoi(clean_wav: np.ndarray, deg_wav: np.ndarray, sr: int = 16000) -> float:
    if pystoi is None:
        return float("nan")
    return float(pystoi.stoi(clean_wav, deg_wav, sr, extended=False))

def calculate_pesq(clean_wav: np.ndarray, deg_wav: np.ndarray, sr: int = 16000) -> float:
    if pesq is None:
        return float("nan")
    try:
        c = np.clip(clean_wav.astype(np.float32), -1.0, 1.0)
        d = np.clip(deg_wav.astype(np.float32), -1.0, 1.0)
        mode = "wb" if sr in (16000, 48000) else "nb"
        return float(pesq(sr, c, d, mode))
    except Exception:
        return float("nan")


@torch.inference_mode()
def evaluate_metrics(model: nn.Module, test_loader, synthesis_matrix: Tensor, filterbank: torch.Tensor, device: torch.device, 
                             sample_rate: int = 16000, n_fft: int = 512, hop_length: int = 128, 
                             win_length: int = 512, max_samples: int = 100) -> Dict[str, float]:
    model.eval()
    synthesis_matrix = synthesis_matrix.to(device)
    window = torch.hann_window(win_length, device=device)
    filterbank = filterbank.to(device)
    metrics = {
        "pesq_noisy": [], "pesq_stage1": [],
        "stoi_noisy": [], "stoi_stage1": [],
        "sisdr_noisy": [], "sisdr_stage1": [],
    }
    count = 0
    print(f"Starting objective evaluation on test set for Stage 1 (SR: {sample_rate} Hz)...")
    ema_norm = CausalEmaNorm(alpha=0.99, eps=1e-5).to(device)
    for batch in test_loader:
        noisy_wav = batch["noisy_wav"].to(device)       # Move to GPU
        clean_target = batch["clean_target"].to(device) # Move to GPU
        
        # 1. Do the heavy STFT math on the blazing-fast GPU
        input_spec = torch.stft(noisy_wav, n_fft=512, hop_length=128, win_length=512, window=window, return_complex=True)
        target_spec = torch.stft(clean_target, n_fft=512, hop_length=128, win_length=512, window=window, return_complex=True)
        
        # 2. Do the ERB math on the GPU
        # Make sure self.filterbank is also moved to the device!
        input_erb = torch.matmul(input_spec.abs().square().transpose(1, 2), filterbank.T)
        input_erb = torch.log10(input_erb + 1e-7).unsqueeze(1)
        
        input_spec = input_spec.transpose(1, 2)
        target_spec = target_spec.transpose(1, 2)

        # Normalize ERB features
        input_erb = ema_norm(input_erb)

        with torch.amp.autocast(device.type):
            predicted_gains = model(input_erb)
            stage1_spec = apply_erb_gains(input_spec, predicted_gains, synthesis_matrix)
            
        # Convert back to waveforms
        noisy_wav = spec_to_wav(input_spec, n_fft, hop_length, win_length, window)
        clean_wav = spec_to_wav(target_spec, n_fft, hop_length, win_length, window)
        stage1_wav = spec_to_wav(stage1_spec, n_fft, hop_length, win_length, window)
        for i in range(clean_wav.shape[0]):
            c_wav = clean_wav[i]
            n_wav = noisy_wav[i]
            s1_wav = stage1_wav[i]
            """SI-SDR"""
            metrics["sisdr_noisy"].append(calculate_si_sdr(n_wav, c_wav))
            metrics["sisdr_stage1"].append(calculate_si_sdr(s1_wav, c_wav))
            """PESQ & STOI"""
            c_np = c_wav.cpu().numpy()
            n_np = n_wav.cpu().numpy()
            s1_np = s1_wav.cpu().numpy()
            # Align lengths perfectly
            min_len = min(len(c_np), len(n_np), len(s1_np))
            c_np, n_np, s1_np = c_np[:min_len], n_np[:min_len], s1_np[:min_len]
            metrics["stoi_noisy"].append(calculate_stoi(c_np, n_np, sample_rate))
            metrics["stoi_stage1"].append(calculate_stoi(c_np, s1_np, sample_rate))
            p_n = calculate_pesq(c_np, n_np, sample_rate)
            p_s1 = calculate_pesq(c_np, s1_np, sample_rate)
            if not np.isnan(p_n):
                metrics["pesq_noisy"].append(p_n)
            if not np.isnan(p_s1):
                metrics["pesq_stage1"].append(p_s1)
            count += 1
            if max_samples and count >= max_samples:
                break
        if max_samples and count >= max_samples:
            break
    summary = {
        # PESQ
        "PESQ Noisy": np.nanmean(metrics["pesq_noisy"]),
        "PESQ Stage 1": np.nanmean(metrics["pesq_stage1"]),
        "Delta-PESQ": np.nanmean(metrics["pesq_stage1"]) - np.nanmean(metrics["pesq_noisy"]),
        
        # STOI
        "STOI Noisy": np.nanmean(metrics["stoi_noisy"]),
        "STOI Stage 1": np.nanmean(metrics["stoi_stage1"]),
        "Delta-STOI": np.nanmean(metrics["stoi_stage1"]) - np.nanmean(metrics["stoi_noisy"]),
        
        # SI-SDR
        "SI-SDR Noisy (dB)": np.nanmean(metrics["sisdr_noisy"]),
        "SI-SDR Stage 1 (dB)": np.nanmean(metrics["sisdr_stage1"]),
        "Delta-SI-SDR (dB)": np.nanmean(metrics["sisdr_stage1"]) - np.nanmean(metrics["sisdr_noisy"]),
    }
    
    return summary

In [ ]:
# checkpoint_path = "/kaggle/input/datasets/quanninhhoang/checkpoint/best_model.ckpt"
checkpoint_path = "/kaggle/working/erb_checkpoints/best_model.ckpt"
# checkpoint_path = "/content/drive/MyDrive/DNS_Data/erb_checkpoints/best_model.ckpt"

checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])

# Call the new Stage 1 specific function
summary = evaluate_metrics(
    model=model,
    test_loader=test_loader,
    synthesis_matrix=synthesis_matrix,
    filterbank=filterbank,
    device=device,
    sample_rate=16000,
    max_samples=150,
)

print(os.linesep + "=" * 60)
print(f"{'Metric':<20} | {'Noisy':<10} | {'Stage 1 (ERB)':<13} | {'Delta (Δ)':<10}")
print("-" * 60)

# PESQ
print(
    f"{'PESQ (1.0 - 4.5)':<20} | "
    f"{summary['PESQ Noisy']:<10.2f} | "
    f"{summary['PESQ Stage 1']:<13.2f} | "
    f"{summary['Delta-PESQ']:<+10.2f}"
)
# STOI
print(
    f"{'STOI (0.0 - 1.0)':<20} | "
    f"{summary['STOI Noisy']:<10.3f} | "
    f"{summary['STOI Stage 1']:<13.3f} | "
    f"{summary['Delta-STOI']:<+10.3f}"
)
# SI-SDR
print(
    f"{'SI-SDR (dB)':<20} | "
    f"{summary['SI-SDR Noisy (dB)']:<10.2f} | "
    f"{summary['SI-SDR Stage 1 (dB)']:<13.2f} | "
    f"{summary['Delta-SI-SDR (dB)']:<+10.2f}"
)
print("=" * 60 + os.linesep)

### 2. Using error computation

In [ ]:
def spectrogram_similarity(predicted_spec, target_spec):
    error = (predicted_spec - target_spec).abs().square().flatten(1).sum(dim=1)
    target_energy = (target_spec.abs().square().flatten(1).sum(dim=1))
    normalized_error = error / (target_energy + 1e-8)
    similarity = (1.0 - normalized_error).clamp(0.0, 1.0)
    return similarity * 100.0

In [ ]:
# checkpoint_path = "/kaggle/working/erb_checkpoints/best_model.ckpt"
#checkpoint_path = "/content/drive/MyDrive/DNS_Data/erb_checkpoints/best_model.ckpt"
checkpoint_path = "/kaggle/input/datasets/quanninhhoang/checkpoint/best_model.ckpt"

checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

In [ ]:
total_similarity = 0.0
total_accuracy_stage1 = 0.0
total_samples = 0
total_stage1_samples = 0

model.eval()

window = torch.hann_window(512).to(device)
filterbank = filterbank.to(device)
synthesis_matrix = synthesis_matrix.to(device)

with torch.inference_mode():
    ema_norm = CausalEmaNorm(alpha=0.99, eps=1e-5).to(device)
    for batch in test_loader:
        noisy_wav = batch["noisy_wav"].to(device)      
        clean_target = batch["clean_target"].to(device) 
        
        input_spec = torch.stft(noisy_wav, n_fft=512, hop_length=128, win_length=512, window=window, return_complex=True)
        target_spec = torch.stft(clean_target, n_fft=512, hop_length=128, win_length=512, window=window, return_complex=True)
        
        # Filterbank is already on the device now
        input_erb = torch.matmul(input_spec.abs().square().transpose(1, 2), filterbank.T)
        input_erb = torch.log10(input_erb + 1e-7).unsqueeze(1)
        
        input_spec = input_spec.transpose(1, 2)
        target_spec = target_spec.transpose(1, 2)
        
        input_erb = ema_norm(input_erb)
        
        with torch.amp.autocast(device.type):
            predicted_gains = model(input_erb)
            stage1_spec = apply_erb_gains(input_spec, predicted_gains, synthesis_matrix)

        similarity_stage1 = spectrogram_similarity(stage1_spec, target_spec)
        total_accuracy_stage1 += similarity_stage1.sum().item()
        total_stage1_samples += similarity_stage1.numel()
        
average = (total_accuracy_stage1 / max(total_stage1_samples, 1)) * 100
print(f"Accuracy: {average:.2f}%")

## Listen a single enhanced audio

In [ ]:
from IPython.display import Audio, display

checkpoint_path = "/kaggle/working/erb_checkpoints/best_model.ckpt"
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

ema_norm = CausalEmaNorm(alpha=0.99, eps=1e-5).to(device)
window = torch.hann_window(512).to(device)
synthesis_matrix = synthesis_matrix.to(device) 

batch = next(iter(test_loader))
noisy_wav_orig = batch["noisy_wav"].to(device)       
clean_wav_orig = batch["clean_target"].to(device) 

# STFT
input_spec = torch.stft(noisy_wav_orig, n_fft=512, hop_length=128, win_length=512, window=window, return_complex=True)

# ERB Math
input_erb = torch.matmul(input_spec.abs().square().transpose(1, 2), filterbank.to(device).T)
input_erb = torch.log10(input_erb + 1e-7).unsqueeze(1)
input_erb = ema_norm(input_erb)

input_spec = input_spec.transpose(1, 2)

# Inference
with torch.no_grad():
    with torch.amp.autocast(device.type):
        predicted_gains = model(input_erb)
        stage1_spec = apply_erb_gains(input_spec, predicted_gains, synthesis_matrix)

enh_wav = spec_to_wav(stage1_spec, n_fft=512, hop_length=128, win_length=512, window=window)

noisy_audio = noisy_wav_orig[0].cpu().numpy()
clean_audio = clean_wav_orig[0].cpu().numpy()
enh_audio = enh_wav[0].cpu().numpy()

sample_rate = 16000 
print("Noisy Audio:")
display(Audio(noisy_audio, rate=sample_rate))

print("Enhanced Audio (Stage 1):")
display(Audio(enh_audio, rate=sample_rate))

print("Clean Target Audio:")
display(Audio(clean_audio, rate=sample_rate))

## Evaluate model on VoiceBank+Demand test set

In [ ]:
from thop import profile
try:
    import pysepm
except ImportError:
    print("Warning: pysepm not found. CSIG, CBAK, COVL will be skipped.")
    pysepm = None

"""1. Parameters and MACs"""
def print_model_complexity(model, device, erb_bins=32, df_bins=96):
    num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total Parameters: {num_params / 1e6:.2f} M")

    # Simulate exactly 1 second of audio (16,000 samples)
    # n_fft=512, hop_length=128 -> 16,000 / 128 + 1 = 126 frames
    frames_1sec = 126
    dummy_erb = torch.randn(1, 1, frames_1sec, erb_bins).to(device)

    try:
        # Calculate MACs (Multiply-Accumulate Operations)
        macs, _ = profile(model, inputs=(dummy_erb,), verbose=False)
        print(f"MACs (per 1 second of audio): {macs / 1e9:.2f} G")
    except Exception as e:
        print(f"Could not calculate MACs: {e}")

"""2. VoiceBank+Demand Evaluation"""
def evaluate_vctk_demand(model, clean_dir, noisy_dir, synthesis_matrix, filterbank, device, sample_rate=16000, df_bins=96, df_order=5):
    model.eval()
    synthesis_matrix = synthesis_matrix.to(device)
    filterbank = filterbank.to(device)
    window = torch.hann_window(512, device=device)

    noisy_files = sorted(glob.glob(os.path.join(noisy_dir, "*.wav")))

    # Added noisy metrics and fixed 'cbank' typo
    metrics = {
        "pesq": [], "stoi": [], "sisdr": [], "csig": [], "cbak": [], "covl": [],
        "noisy_pesq": [], "noisy_stoi": [], "noisy_sisdr": [], "noisy_csig": [], "noisy_cbak": [], "noisy_covl": []
    }

    print(os.linesep + f"Evaluating on {len(noisy_files)} files from VoiceBank+DEMAND...")

    ema_norm = CausalEmaNorm(alpha=0.99, eps=1e-5).to(device)
    with torch.inference_mode():
        for i, noisy_path in enumerate(noisy_files):
            filename = os.path.basename(noisy_path)
            clean_path = os.path.join(clean_dir, filename)

            if not os.path.exists(clean_path):
                continue
            
            # Load and resample
            noisy_wav, sr_n = torchaudio.load(noisy_path)
            clean_wav, sr_c = torchaudio.load(clean_path)

            if sr_n != sample_rate:
                noisy_wav = torchaudio.functional.resample(noisy_wav, sr_n, sample_rate)
            if sr_c != sample_rate:
                clean_wav = torchaudio.functional.resample(clean_wav, sr_c, sample_rate)
                
            noisy_wav = noisy_wav.mean(dim=0).to(device)
            clean_wav = clean_wav.mean(dim=0).to(device)

            # Align lengths perfectly
            min_len = min(noisy_wav.shape[-1], clean_wav.shape[-1])
            noisy_wav = noisy_wav[:min_len]
            clean_wav = clean_wav[:min_len]
            
            # Pre-processing
            input_spec = torch.stft(noisy_wav, n_fft=512, hop_length=128, win_length=512, window=window, return_complex=True).transpose(0, 1).unsqueeze(0).to(device)
            
            # Calculate ERB features
            input_erb = torch.matmul(input_spec.abs().square(), filterbank.T)
            input_erb = torch.log10(input_erb + 1e-7).unsqueeze(1)
            
            input_erb = ema_norm(input_erb)
            
            
            # Model Forward Pass
            with torch.amp.autocast(device.type):
                predicted_gains = model(input_erb)
                stage1_spec = apply_erb_gains(input_spec, predicted_gains, synthesis_matrix)
            
            # Inverse STFT
            enh_wav = spec_to_wav(stage1_spec, n_fft=512, hop_length=128, win_length=512, window=window).squeeze()
            
            # CPU conversions
            c_np = clean_wav.cpu().numpy()
            e_np = enh_wav.cpu().numpy()
            n_np = noisy_wav.cpu().numpy()
            
            # Align lengths after Inverse STFT reconstruction
            min_out_len = min(len(c_np), len(e_np))
            c_np = c_np[:min_out_len]
            e_np = e_np[:min_out_len]
            n_np = n_np[:min_out_len]

            # ----------------------------------
            # Calculate Enhanced Metrics
            # ----------------------------------
            metrics["pesq"].append(calculate_pesq(c_np, e_np, sample_rate))
            metrics["stoi"].append(calculate_stoi(c_np, e_np, sample_rate))
            metrics["sisdr"].append(calculate_si_sdr(torch.tensor(e_np), torch.tensor(c_np)))
            
            # ----------------------------------
            # Calculate Noisy Baseline Metrics
            # ----------------------------------
            metrics["noisy_pesq"].append(calculate_pesq(c_np, n_np, sample_rate))
            metrics["noisy_stoi"].append(calculate_stoi(c_np, n_np, sample_rate))
            metrics["noisy_sisdr"].append(calculate_si_sdr(torch.tensor(n_np), torch.tensor(c_np)))
            
            if pysepm is not None:
                try:
                    csig, cbak, covl = pysepm.composite(c_np, e_np, sample_rate)
                    metrics["csig"].append(csig)
                    metrics["cbak"].append(cbak)
                    metrics["covl"].append(covl)
                    
                    n_csig, n_cbak, n_covl = pysepm.composite(c_np, n_np, sample_rate)
                    metrics["noisy_csig"].append(n_csig)
                    metrics["noisy_cbak"].append(n_cbak)
                    metrics["noisy_covl"].append(n_covl)
                except Exception:
                    pass
                
            if (i+1) % 50 == 0:
                print(f"Processed {i+1}/{len(noisy_files)} files...")

    # ==========================================
    # Print Table Summary
    # ==========================================
    print("\n" + "="*50)
    print(" VoiceBank+DEMAND Test Set Results")
    print("="*50)
    print(f"{'Metric':<10} | {'Noisy':<8} | {'Enhanced':<8} | {'Delta (Δ)':<8}")
    print("-" * 50)
    
    p_n, p_e = np.nanmean(metrics['noisy_pesq']), np.nanmean(metrics['pesq'])
    s_n, s_e = np.nanmean(metrics['noisy_stoi']), np.nanmean(metrics['stoi'])
    si_n, si_e = np.nanmean(metrics['noisy_sisdr']), np.nanmean(metrics['sisdr'])
    
    print(f"{'PESQ':<10} | {p_n:<8.3f} | {p_e:<8.3f} | {p_e - p_n:<+8.3f}")
    print(f"{'STOI':<10} | {s_n:<8.3f} | {s_e:<8.3f} | {s_e - s_n:<+8.3f}")
    print(f"{'SI-SDR':<10} | {si_n:<8.2f} | {si_e:<8.2f} | {si_e - si_n:<+8.2f}")
    
    if pysepm is not None and len(metrics["csig"]) > 0:
        cs_n, cs_e = np.nanmean(metrics['noisy_csig']), np.nanmean(metrics['csig'])
        cb_n, cb_e = np.nanmean(metrics['noisy_cbak']), np.nanmean(metrics['cbak'])
        co_n, co_e = np.nanmean(metrics['noisy_covl']), np.nanmean(metrics['covl'])
        
        print(f"{'CSIG':<10} | {cs_n:<8.3f} | {cs_e:<8.3f} | {cs_e - cs_n:<+8.3f}")
        print(f"{'CBAK':<10} | {cb_n:<8.3f} | {cb_e:<8.3f} | {cb_e - cb_n:<+8.3f}")
        print(f"{'COVL':<10} | {co_n:<8.3f} | {co_e:<8.3f} | {co_e - co_n:<+8.3f}")
    print("="*50)

In [ ]:
import numpy as np

# ----------------------------------------------------
# HOTFIX for NumPy 2.0 compatibility with pysepm
# ----------------------------------------------------
if not hasattr(np, 'NaN'):
    np.NaN = np.nan
if not hasattr(np, 'NINF'):
    np.NINF = -np.inf
# ----------------------------------------------------

In [ ]:
checkpoint_path = "/content/drive/MyDrive/DNS_Data/erb_checkpoints/best_model.ckpt"
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

In [ ]:
clean_dir="/kaggle/input/datasets/quanninhhoang/voicebank-demand/clean_testset_wav/clean_testset_wav"
noisy_dir="/kaggle/input/datasets/quanninhhoang/voicebank-demand/noisy_testset_wav/noisy_testset_wav"

print_model_complexity(model, device)

evaluate_vctk_demand(
    model=model,
    clean_dir=clean_dir,
    noisy_dir=noisy_dir,
    synthesis_matrix=synthesis_matrix,
    filterbank=filterbank,
    device=device,
)

## Listen to VoiceBank+Demand audio

In [ ]:
import IPython.display as ipd
import glob

def listen_to_samples(model, clean_dir, noisy_dir, synthesis_matrix, filterbank, device, num_samples=3):
    noisy_files = sorted(glob.glob(os.path.join(noisy_dir, "*.wav")))
    
    # Pick a few random files to listen to
    sample_files = random.sample(noisy_files, min(num_samples, len(noisy_files)))
    
    window = torch.hann_window(512, device=device)
    synthesis_matrix = synthesis_matrix.to(device)
    filterbank = filterbank.to(device)
    
    model.eval()

    ema_norm = CausalEmaNorm(alpha=0.99, eps=1e-5).to(device)
    with torch.inference_mode():
        for noisy_path in sample_files:
            filename = os.path.basename(noisy_path)
            clean_path = os.path.join(clean_dir, filename)
            
            if not os.path.exists(clean_path):
                continue
                
            print(f"\n{'='*50}")
            print(f"File: {filename}")
            print(f"{'='*50}")
            
            # Load and resample
            noisy_wav, sr_n = torchaudio.load(noisy_path)
            clean_wav, sr_c = torchaudio.load(clean_path)
            
            if sr_n != 16000:
                noisy_wav = torchaudio.functional.resample(noisy_wav, sr_n, 16000)
            if sr_c != 16000:
                clean_wav = torchaudio.functional.resample(clean_wav, sr_c, 16000)
                
            noisy_wav = noisy_wav.mean(dim=0).to(device)
            clean_wav = clean_wav.mean(dim=0).to(device)

            min_len = min(noisy_wav.shape[-1], clean_wav.shape[-1])
            noisy_wav = noisy_wav[:min_len]
            clean_wav = clean_wav[:min_len]

            # ---------------------------
            # Enhance Audio
            # ---------------------------
            input_spec = torch.stft(noisy_wav, n_fft=512, hop_length=128, win_length=512, window=window, return_complex=True).transpose(0, 1).unsqueeze(0).to(device)
            
            input_erb = torch.matmul(input_spec.abs().square(), filterbank.T)
            input_erb = torch.log10(input_erb + 1e-7).unsqueeze(1)
            
            input_erb = ema_norm(input_erb)
            
            
            with torch.amp.autocast(device.type):
                predicted_gains = model(input_erb)
                stage1_spec = apply_erb_gains(input_spec, predicted_gains, synthesis_matrix)
            
            enh_wav = spec_to_wav(stage1_spec, n_fft=512, hop_length=128, win_length=512, window=window).squeeze()
            
            # ---------------------------
            # Display Playable Audio
            # ---------------------------
            noisy_cpu = noisy_wav.cpu().numpy()
            clean_cpu = clean_wav.cpu().numpy()
            enh_cpu = enh_wav.cpu().numpy()
            
            print("1. Noisy Input:")
            display(ipd.Audio(noisy_cpu, rate=16000))
            
            print("2. Enhanced Output:")
            display(ipd.Audio(enh_cpu, rate=16000))
            
            print("3. Clean Ground Truth:")
            display(ipd.Audio(clean_cpu, rate=16000))



In [ ]:
# clean_dir="/kaggle/input/datasets/quanninhhoang/voicebank-demand/clean_testset_wav/clean_testset_wav"
# noisy_dir="/kaggle/input/datasets/quanninhhoang/voicebank-demand/noisy_testset_wav/noisy_testset_wav"
clean_dir="/content/drive/MyDrive/DNS_Data/VoiceBank_Demand/clean_testset_wav/"
noisy_dir="/content/drive/MyDrive/DNS_Data/VoiceBank_Demand/noisy_testset_wav/"
listen_to_samples(
    model=model, 
    clean_dir=clean_dir, 
    noisy_dir=noisy_dir, 
    synthesis_matrix=synthesis_matrix, 
    filterbank=filterbank, 
    device=device,
    num_samples=6  
)

In [ ]:
import zipfile
import os
from IPython.display import FileLink

file_to_zip = "/kaggle/working/erb_checkpoints/best_model.ckpt"
zip_filename = 'best_checkpoint.zip'

if os.path.exists(file_to_zip):
    # 1. Create a zip file and add only the best checkpoint to it
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(file_to_zip)
    
    print("Zip created successfully!")
    
    # 2. Generate the download link
    display(FileLink(zip_filename))
else:
    print(f"Error: {file_to_zip} does not exist in the current directory.")